In [1]:
import requests
import pandas as pd
import json
import smtplib
import os

In [3]:
stock_api_key = 'AMKNC2TP5I7XXAN7'
news_api_key = 'abc345'#not required as you will see further down
symbol='TSLA'

In [4]:
# making the API call to get stock data
url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={stock_api_key}"
r = requests.get(url)
data = r.json()

In [5]:
#converting to a df for further use
df = pd.DataFrame(data['Time Series (Daily)'])
df.head(2)

,2026-03-23,2026-03-20,2026-03-19,2026-03-18,2026-03-17,2026-03-16,2026-03-13,2026-03-12,2026-03-11,2026-03-10,...,2025-11-10,2025-11-07,2025-11-06,2025-11-05,2025-11-04,2025-11-03,2025-10-31,2025-10-30,2025-10-29,2025-10-28
1. open,373.0900,379.8500,387.2700,399.0000,395.6900,396.2200,399.1650,405.1750,402.2800,402.2200,...,439.6000,437.9200,461.9600,452.0500,454.4600,455.9900,446.7500,451.0500,462.5000,454.7750
2. high,385.3300,379.8900,387.2700,403.0650,400.1200,403.7300,400.2000,406.5000,416.3799,406.5900,...,449.6715,439.3600,467.4500,466.3299,460.2200,474.0700,458.0000,455.0607,465.7000,467.0000


In [6]:
#transposing the data to make it readable
df=df.T
df.head(2)

,1. open,2. high,3. low,4. close,5. volume
2026-03-23,373.0900,385.3300,372.7300,380.8500,74606049
2026-03-20,379.8500,379.8900,364.4601,367.9600,78628603


In [7]:
#sorting the index in ascending order
df=df.sort_index(ascending=True)
#renaming columns
df.columns = ['Open','High','Low','Close','Volume']
df.head(2)

,Open,High,Low,Close,Volume
2025-10-28,454.7750,467.0000,451.6000,460.5500,80185667
2025-10-29,462.5000,465.7000,452.6500,461.5100,67983544


In [8]:
#convert all the column dataypes to numeric to do calculations
df = df.apply(pd.to_numeric)

#creating a new column to calculate the closing price difference using diff() 
df['closing_difference'] = df['Close'].diff()

df.head()

,Open,High,Low,Close,Volume,closing_difference
2025-10-28,454.775,467.0000,451.6000,460.55,80185667,NaN
2025-10-29,462.500,465.7000,452.6500,461.51,67983544,0.96
2025-10-30,451.050,455.0607,439.6100,440.10,72447938,-21.41
2025-10-31,446.750,458.0000,443.6855,456.56,83135787,16.46
2025-11-03,455.990,474.0700,453.8000,468.37,84595244,11.81


In [9]:
#changing the value of the last row from NaN to 0
#diff() will calculate the the difference w.r.t the previous row. First row has no previous value and hence will always be NaN
#replacing the NaN with 0
df['closing_difference'].fillna(0,inplace=True)

In [10]:
#same analogy with pct_change. First change will always be Nan
df['closing_pct_change'] = round(df['Close'].pct_change()*100,2)
#filling in the empty row value with 0
df['closing_pct_change'].fillna(0,inplace=True)

df.head()

,Open,High,Low,Close,Volume,closing_difference,closing_pct_change
2025-10-28,454.775,467.0000,451.6000,460.55,80185667,0.00,0.00
2025-10-29,462.500,465.7000,452.6500,461.51,67983544,0.96,0.21
2025-10-30,451.050,455.0607,439.6100,440.10,72447938,-21.41,-4.64
2025-10-31,446.750,458.0000,443.6855,456.56,83135787,16.46,3.74
2025-11-03,455.990,474.0700,453.8000,468.37,84595244,11.81,2.59


In [11]:
#getting the abs value of the change. You want to be notified of a swing > 5%, whether its +ve or -ve
df['absolute_change'] = abs(df['closing_pct_change'])

df.head()

,Open,High,Low,Close,Volume,closing_difference,closing_pct_change,absolute_change
2025-10-28,454.775,467.0000,451.6000,460.55,80185667,0.00,0.00,0.00
2025-10-29,462.500,465.7000,452.6500,461.51,67983544,0.96,0.21,0.21
2025-10-30,451.050,455.0607,439.6100,440.10,72447938,-21.41,-4.64,4.64
2025-10-31,446.750,458.0000,443.6855,456.56,83135787,16.46,3.74,3.74
2025-11-03,455.990,474.0700,453.8000,468.37,84595244,11.81,2.59,2.59


In [12]:
#use reset_index to have the dates as a column.
#this will make it easier to use as a ref when making api calls
df.reset_index()

,index,Open,High,Low,Close,Volume,closing_difference,closing_pct_change,absolute_change
0,2025-10-28,454.775,467.0000,451.6000,460.55,80185667,0.00,0.00,0.00
1,2025-10-29,462.500,465.7000,452.6500,461.51,67983544,0.96,0.21,0.21
2,2025-10-30,451.050,455.0607,439.6100,440.10,72447938,-21.41,-4.64,4.64
3,2025-10-31,446.750,458.0000,443.6855,456.56,83135787,16.46,3.74,3.74
4,2025-11-03,455.990,474.0700,453.8000,468.37,84595244,11.81,2.59,2.59
...,...,...,...,...,...,...,...,...,...
95,2026-03-17,395.690,400.1200,393.0000,399.27,46890467,3.71,0.94,0.94
96,2026-03-18,399.000,403.0650,392.3100,392.78,50853149,-6.49,-1.63,1.63
97,2026-03-19,387.270,387.2700,378.7300,380.30,67078259,-12.48,-3.18,3.18
98,2026-03-20,379.850,379.8900,364.4601,367.96,78628603,-12.34,-3.24,3.24


In [13]:
#using inplace argument to modify the df
df.reset_index(inplace=True)

#renaming the column
df.rename(columns={'index': 'Date'},inplace=True)

df.head()

,Date,Open,High,Low,Close,Volume,closing_difference,closing_pct_change,absolute_change
0,2025-10-28,454.775,467.0000,451.6000,460.55,80185667,0.00,0.00,0.00
1,2025-10-29,462.500,465.7000,452.6500,461.51,67983544,0.96,0.21,0.21
2,2025-10-30,451.050,455.0607,439.6100,440.10,72447938,-21.41,-4.64,4.64
3,2025-10-31,446.750,458.0000,443.6855,456.56,83135787,16.46,3.74,3.74
4,2025-11-03,455.990,474.0700,453.8000,468.37,84595244,11.81,2.59,2.59


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                100 non-null    object 
 1   Open                100 non-null    float64
 2   High                100 non-null    float64
 3   Low                 100 non-null    float64
 4   Close               100 non-null    float64
 5   Volume              100 non-null    int64  
 6   closing_difference  100 non-null    float64
 7   closing_pct_change  100 non-null    float64
 8   absolute_change     100 non-null    float64
dtypes: float64(7), int64(1), object(1)
memory usage: 7.2+ KB


In [15]:
#changing the Date data type to str to parse into api calls
df['Date'] = df['Date'].astype(str)

#this is how you would filter rows to get a date 
#first part of the syntax: creating a df where the rows are filtered whereever the absolute change is > 5
#second part the syntax: getting the first date value from the filtered rows
date = df[df['absolute_change']>=5.00].iloc[0,0]
date

'2025-11-04'

                                                      ## Making API calls ## 

In [17]:
#https://newsapi.org/v2/everything
#news api service provided by the course only provides new data going back the previous day. You cannot
#access historical data. This will not work
response = requests.get(f'https://newsapi.org/v2/everything?q=tesla&from=2026-02-01&to=2026-02-03&sortBy=popularity&apiKey=abc678')
data=response.json()
data

{'status': 'error',
 'code': 'parameterInvalid',
 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-23, but you have requested 2026-02-01. You may need to upgrade to a paid plan.'}

In [39]:
#using a different service: https://gnews.io/
#this will at least give you new going back 30 days

# response = requests.get(f'https://gnews.io/api/v4/search?q=Google&lang=en&max=5&apikey={news_api_key}&from=2026-03-20T05:09:00Z')
# data=response.json()
# data

"""
This was not giving historical data despite specifying a past date in the url

""";

In [36]:
# from gnews import GNews

# # Initialize GNews with various parameters, including proxy
# google_news = GNews(
#     language='en',
#     country='US',
# #     period='7d',
#     start_date=(2026,3,20),
#     end_date=(2026,3,23),
#     max_results=10
# )

# google_news.get_news('India')


"""
This is also not working.

""";

                     Since none of the API calls are working and I cannot find one that can provide free historical data,
                     this is what I'm doing instead:
                     1. Check for absolute changes > $3.
                     2. Send an alert to your email about the change.
                     3. Use GitHub actions to run this everyday.

In [18]:
def send_alert():
    from_email = 'krishnanrahul929@gmail.com'
    to_email = 'rahulakrish@gmail.com'
    password = 'abc123'

    with smtplib.SMTP("smtp.gmail.com",587) as connection:
        connection.starttls()
        connection.login(user=from_email, password=password)
        connection.sendmail(from_addr = from_email,
                            to_addrs = to_email,
                           msg = 'Subject: TSLA alert\n\nChange from the previous day > $3.\nFind out why')


In [20]:

###################### this will not run at work becuase of network firewalls ###################################

#get the last value in the absolute change column.
#this will correspond to the previous day's closing change.
if df.iloc[-1,-1] >= 1:
    send_alert()   

In [23]:
%%writefile tesla_stock_alert.py

#this file will be used to run a GitHub workflow

import requests
import pandas as pd
import json
import smtplib
import os

stock_api_key = os.environ.get('STOCK_API_KEY')
news_api_key = 'abc345'
symbol='TSLA'


# making the API call to get stock data
url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={stock_api_key}"
r = requests.get(url)
data = r.json()

#converting to a df for further use
df = pd.DataFrame(data['Time Series (Daily)'])

#transposing the data to make it readable
df=df.T

#sorting the index in ascending order
df=df.sort_index(ascending=True)
#renaming columns
df.columns = ['Open','High','Low','Close','Volume']


#convert all the column dataypes to numeric to do calculations
df = df.apply(pd.to_numeric)

#creating a new column to calculate the closing price difference using diff() 
df['closing_difference'] = df['Close'].diff()

#changing the value of the last row from NaN to 0
#diff() will calculate the the difference w.r.t the previous row. First row has no previous value and hence will always be NaN
#replacing the NaN with 0
df['closing_difference'].fillna(0,inplace=True)

#same analogy with pct_change. First change will always be Nan
df['closing_pct_change'] = round(df['Close'].pct_change()*100,2)
#filling in the empty row value with 0
df['closing_pct_change'].fillna(0,inplace=True)


#getting the abs value of the change. You want to be notified of a swing > 5%, whether its +ve or -ve
df['absolute_change'] = abs(df['closing_pct_change'])

#use reset_index to have the dates as a column.
#this will make it easier to use as a ref when making api calls
df.reset_index()

#using inplace argument to modify the df
df.reset_index(inplace=True)

#renaming the column
df.rename(columns={'index': 'Date'},inplace=True)

#changing the Date data type to str to parse into api calls
df['Date'] = df['Date'].astype(str)


def send_alert():
    from_email = os.environ.get('FROM_EMAIL')
    to_email = os.environ.get('TO_EMAIL')
    password = os.environ.get('PASSWORD')

    with smtplib.SMTP("smtp.gmail.com",587) as connection:
        connection.starttls()
        connection.login(user=from_email, password=password)
        connection.sendmail(from_addr = from_email,
                            to_addrs = to_email,
                            msg = 'Subject: TSLA alert\n\nChange from the previous day > $3.\nFind out why')
        
        
#get the last value in the absolute change column.
#this will correspond to the previous day's closing change.
if df.iloc[-1,-1] >= 1:
    send_alert()   


Writing tesla_stock_alert.py
